In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


# ============================================================================
# 10. FULL COMPARISON: SYSTEM x TEXT-CONDITION x PROMPT-FORMAT
#     RAW vs. CLEAN (11) vs. DOUBLY-CLEAN (11 + 12)
# ============================================================================
# Two independent audits have now found confounds in the full-text ICLE
# evaluation:
#   11 -- essays that literally self-disclose their own country/language
#         (e.g. "I am from Spain"). ~23% of the eval subset.
#   12 -- essays whose topic/prompt is disproportionately (>=80%) written by
#         a single L1 in the wider ICLE corpus (Brooke & Hirst 2012's
#         finding, confirmed here for an LLM). On the CLEAN subset from 11,
#         topic-exclusive essays still hit ~98% accuracy vs. ~65% for the
#         rest.
# These are complementary, not redundant (the per-language patterns differ:
# JPN/KOR dominate the self-disclosure leak, DEU/JPN/KOR dominate the topic
# leak). This notebook now reports THREE tiers per condition:
#   raw          -- all eval-subset rows (matches 04/05/07/08/09 directly).
#   clean        -- self-disclosing essays excluded (11 only).
#   doubly_clean -- ALSO excludes topic-exclusive ICLE essays (11 + 12).
#                   TOEFL11 rows are unaffected by this second filter, since
#                   12's topic metadata is ICLE-specific (TOEFL11 was
#                   designed with shared prompts across L1 groups).
# doubly_clean is the most defensible "does this system do real NLI" number
# available from this pipeline. The gaps between raw -> clean -> doubly_clean
# are themselves the quantified size of each confound.
#
# Requires 11 and 12 to have been run at least once for the CLEAN and
# DOUBLY-CLEAN tiers. Missing inputs are skipped with a message, not a
# crash -- this notebook does not need every upstream notebook to be
# complete to produce partial results.
#
# NEW: also cross-checks the doubly_clean tier against notebook 15's
# institution-self-mention audit, exporting a per-essay audit_matrix.csv
# for the LLM/fulltext/forced_json/doubly_clean/ICLE slice specifically.
# This reuses the in-memory `subset_llm` dataframe from the loop below
# rather than re-reading predictions from disk, so the triple-clean number
# it prints is guaranteed consistent with the doubly_clean number printed
# for that same arm -- if the two ever disagree, that is a sign of a stale
# file on disk, not a bug in this notebook.

BASE_DIR = Path("..").resolve()
RESULTS_DIR = BASE_DIR / "data/processed/results"
EVAL_SUBSET_PATH = RESULTS_DIR / "llm_eval_subset.csv"
LEAKAGE_FLAGGED_PATH = RESULTS_DIR / "leakage_flagged_essays.csv"
TOPIC_FLAGS_PATH = RESULTS_DIR / "icle_topic_flags.csv"
INSTITUTION_FLAGS_PATH = RESULTS_DIR / "institution_leak_flagged_essays.csv"

MASKED_MATRICES_PATH = BASE_DIR / "data/processed/nli_feature_matrices.pkl"
FULLTEXT_MATRICES_PATH = BASE_DIR / "data/processed/nli_feature_matrices_fulltext.pkl"
MASKED_SVM_MODEL_PATH = BASE_DIR / "models/linear_svc_nli_model.pkl"
FULLTEXT_SVM_MODEL_PATH = BASE_DIR / "models/linear_svc_nli_model_fulltext.pkl"

OUTPUT_METRICS_PATH = RESULTS_DIR / "full_2x2_comparison_metrics.csv"
OUTPUT_DELTA_PATH = RESULTS_DIR / "full_2x2_shortcut_deltas.csv"
OUTPUT_AUDIT_MATRIX_PATH = RESULTS_DIR / "audit_matrix.csv"


def svm_predictions_on_subset(matrices_path: Path, model_path: Path, eval_df: pd.DataFrame) -> pd.Series:
    with matrices_path.open("rb") as f:
        data = pickle.load(f)
    with model_path.open("rb") as f:
        model = pickle.load(f)

    doc_to_index = {doc_id: i for i, doc_id in enumerate(data["doc_id"])}
    missing = sorted(set(eval_df["doc_id"]) - set(doc_to_index))
    if missing:
        raise ValueError(f"doc_ids missing from {matrices_path.name}: {missing[:5]}")

    indices = np.array([doc_to_index[d] for d in eval_df["doc_id"]])
    predictions = model.predict(data["X"][indices])
    return pd.Series(predictions, index=eval_df.index)


def metric_row(y_true, y_pred, system, text_condition, prompt_format, partition, subset, n_dropped=0) -> dict:
    return {
        "system": system,
        "text_condition": text_condition,
        "prompt_format": prompt_format,
        "subset": subset,  # "raw", "clean", or "doubly_clean"
        "partition": partition,
        "n": len(y_true),
        "n_dropped_parse_or_api_errors": n_dropped,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


eval_df = pd.read_csv(EVAL_SUBSET_PATH)

self_referential = pd.Series(False, index=eval_df.index)
if LEAKAGE_FLAGGED_PATH.exists():
    flagged = pd.read_csv(LEAKAGE_FLAGGED_PATH)
    self_ref_ids = set(flagged.loc[flagged["self_referential_in_text"] == True, "doc_id"])
    self_referential = eval_df["doc_id"].isin(self_ref_ids)
    print(f"{int(self_referential.sum())} / {len(eval_df)} rows self-disclose their own country/language (11).")
else:
    print(f"WARNING: {LEAKAGE_FLAGGED_PATH.name} not found -- run 11 first. CLEAN tier will equal RAW.")

topic_exclusive = pd.Series(False, index=eval_df.index)
if TOPIC_FLAGS_PATH.exists():
    topic_flags = pd.read_csv(TOPIC_FLAGS_PATH)
    topic_ids = set(topic_flags.loc[topic_flags["topic_exclusive_to_own_label"] == True, "doc_id"])
    topic_exclusive = eval_df["doc_id"].isin(topic_ids)
    print(f"{int(topic_exclusive.sum())} / {len(eval_df)} rows have a topic exclusive to their own label (12).")
else:
    print(f"WARNING: {TOPIC_FLAGS_PATH.name} not found -- run 12 first. DOUBLY_CLEAN tier will equal CLEAN.")

eval_df["self_referential_in_text"] = self_referential
eval_df["topic_exclusive_to_own_label"] = topic_exclusive

SUBSETS = {
    "raw": eval_df,
    "clean": eval_df[~eval_df["self_referential_in_text"]],
    "doubly_clean": eval_df[~eval_df["self_referential_in_text"] & ~eval_df["topic_exclusive_to_own_label"]],
}
for name, subset_df in SUBSETS.items():
    print(f"  {name}: {len(subset_df)} rows")

all_rows = []

# --- SVM on the matched subsample ---
for text_condition, matrices_path, model_path in [
    ("masked", MASKED_MATRICES_PATH, MASKED_SVM_MODEL_PATH),
    ("fulltext", FULLTEXT_MATRICES_PATH, FULLTEXT_SVM_MODEL_PATH),
]:
    if not (matrices_path.exists() and model_path.exists()):
        which_nb = "04_model_training.ipynb" if text_condition == "masked" else "07_unmasked_svm_baseline.ipynb"
        print(f"Skipping SVM/{text_condition}: artifacts not found. Run {which_nb} first.")
        continue

    svm_pred_full = svm_predictions_on_subset(matrices_path, model_path, eval_df)

    for subset_name, subset_df in SUBSETS.items():
        svm_pred = svm_pred_full.loc[subset_df.index]
        for partition, part_df in subset_df.groupby("eval_partition"):
            all_rows.append(
                metric_row(
                    part_df["label"], svm_pred.loc[part_df.index], "SVM", text_condition, "n/a", partition, subset_name
                )
            )
        all_rows.append(metric_row(subset_df["label"], svm_pred, "SVM", text_condition, "n/a", "all", subset_name))

# --- LLM arms ---
llm_sources = [
    ("LLM", "masked", "forced_json", RESULTS_DIR / "llm_predictions.csv", "05_llm_evaluation.ipynb"),
    ("LLM", "fulltext", "forced_json", RESULTS_DIR / "llm_predictions_fulltext.csv", "08_llm_full_text_baseline.ipynb"),
    ("LLM", "masked", "reasoning", RESULTS_DIR / "llm_predictions_masked_reasoning.csv", "09_llm_reasoning_ablation.ipynb"),
    ("LLM", "fulltext", "reasoning", RESULTS_DIR / "llm_predictions_fulltext_reasoning.csv", "09_llm_reasoning_ablation.ipynb"),
]

for system, text_condition, prompt_format, path, which_nb in llm_sources:
    if not path.exists():
        print(f"Skipping {system}/{text_condition}/{prompt_format}: {path.name} not found. Run {which_nb} first.")
        continue
    df_llm = pd.read_csv(path)
    n_total = len(df_llm)
    df_llm = df_llm[df_llm["error"].fillna("").eq("") & df_llm["predicted_l1"].notna()].copy()
    n_dropped_errors = n_total - len(df_llm)

    if len(df_llm) < len(eval_df):
        print(f"NOTE: {path.name} has {len(df_llm)} / {len(eval_df)} eval-subset rows -- treating as a partial run.")

    for subset_name, subset_df in SUBSETS.items():
        subset_llm = df_llm[df_llm["doc_id"].isin(subset_df["doc_id"])]
        for partition, part_df in subset_llm.groupby("eval_partition"):
            all_rows.append(
                metric_row(
                    part_df["true_l1"], part_df["predicted_l1"], system, text_condition, prompt_format,
                    partition, subset_name, n_dropped_errors,
                )
            )
        all_rows.append(
            metric_row(
                subset_llm["true_l1"], subset_llm["predicted_l1"], system, text_condition, prompt_format,
                "all", subset_name, n_dropped_errors,
            )
        )

        # --- Triple-clean cross-check (10 + 11 + 12 + 15) ---
        # Only meaningful for the arm that produces the paper's headline
        # doubly_clean ICLE number (LLM, fulltext, forced_json). Reuses
        # `subset_llm` directly rather than re-reading predictions from
        # disk, so it cannot silently diverge from the doubly_clean row
        # printed for this same arm in the table below.
        if (
            system == "LLM"
            and text_condition == "fulltext"
            and prompt_format == "forced_json"
            and subset_name == "doubly_clean"
        ):
            if INSTITUTION_FLAGS_PATH.exists():
                inst_flags = pd.read_csv(INSTITUTION_FLAGS_PATH)
                inst_ids = set(inst_flags["doc_id"])

                icle_dc = subset_llm[subset_llm["eval_partition"] == "iclev3_external_test"].copy()
                icle_dc["institution_flag"] = icle_dc["doc_id"].isin(inst_ids)
                icle_dc["correct"] = icle_dc["true_l1"] == icle_dc["predicted_l1"]
                icle_dc.to_csv(OUTPUT_AUDIT_MATRIX_PATH, index=False)

                print(
                    f"\n[sanity check] doubly_clean ICLE n={len(icle_dc)}, "
                    f"accuracy={icle_dc['correct'].mean():.6f} "
                    f"(must match the LLM/fulltext/forced_json/doubly_clean/iclev3_external_test "
                    f"row in the table below -- if it doesn't, a stale file is being read somewhere)"
                )
                triple_clean = icle_dc[~icle_dc["institution_flag"]]
                print(
                    f"[triple-clean, 11+12+15] n={len(triple_clean)}, "
                    f"accuracy={triple_clean['correct'].mean():.6f}"
                )
                print(f"Saved: {OUTPUT_AUDIT_MATRIX_PATH}")
            else:
                print(f"WARNING: {INSTITUTION_FLAGS_PATH.name} not found -- run 15 first for the triple-clean check.")

metrics = pd.DataFrame(all_rows)
metrics.to_csv(OUTPUT_METRICS_PATH, index=False)
print("\n=== Full comparison table (RAW / CLEAN / DOUBLY_CLEAN) ===")
print(metrics.sort_values(["subset", "system", "prompt_format", "partition", "text_condition"]).to_string(index=False))

# --- Shortcut deltas: accuracy(fulltext) - accuracy(masked), per system/prompt_format/partition/subset ---
pivot = metrics.pivot_table(
    index=["subset", "system", "prompt_format", "partition"], columns="text_condition", values="accuracy"
).reset_index()

if "fulltext" in pivot.columns and "masked" in pivot.columns:
    pivot["shortcut_delta_accuracy"] = pivot["fulltext"] - pivot["masked"]
    pivot.to_csv(OUTPUT_DELTA_PATH, index=False)
    print("\n=== Shortcut deltas (fulltext accuracy - masked accuracy), RAW / CLEAN / DOUBLY_CLEAN ===")
    print(pivot.sort_values(["subset", "system", "prompt_format", "partition"]).to_string(index=False))
    print(
        "\nraw -> clean is the portion of the delta explained by literal self-disclosure (11). "
        "clean -> doubly_clean is the portion additionally explained by topic exclusivity in ICLE (12). "
        "Whatever remains at doubly_clean is the best current estimate of genuine lexical/structural signal."
    )
else:
    print("Not enough conditions present yet to compute shortcut deltas -- run the missing notebooks above.")

print(f"\nSaved: {OUTPUT_METRICS_PATH}")
print(f"Saved: {OUTPUT_DELTA_PATH}")
